# 05 Open-Meteo API and Kafka Producer

## Purpose
Fetch current Open-Meteo air-quality data for the selected European cities, preserve raw JSON in Bronze, build versioned events, and optionally send them to a group-specific Kafka topic.

## Inputs
- `data/silver/city_reference.parquet`
- Open-Meteo Air Quality REST API
- `.env` values for the optional Kafka producer

## Outputs
- `data/bronze/open_meteo_raw/<city_id>.json`
- `data/bronze/open_meteo_raw/open_meteo_air_quality_events.jsonl`
- `data/bronze/open_meteo_raw/open_meteo_ingestion_manifest.json`
- Kafka events on `KAFKA_TOPIC_AIR_QUALITY_LIVE` when FH Kafka is enabled
- `data/samples/open_meteo_phase5_events_sample.jsonl`
- Local mock broker JSONL for reproducible offline testing

## Technologies used
Python, requests, kafka-python, JSON.

## Configuration
Set `RUN_OPEN_METEO_API_FETCH=false` to reuse existing local Bronze JSON. Set `RUN_OPEN_METEO_KAFKA_PRODUCER=true` on FH JupyterHub to publish to Kafka. `KAFKA_MODE=auto` tries FH Kafka and falls back to a transparent local JSONL mock broker when allowed. Safe local runs with the producer disabled use the mock broker directly.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
PROJECT_ROOT = Path.cwd()
DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))
CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
OPEN_METEO_RAW_DIR = DATA_DIR / "bronze" / "open_meteo_raw"
SAMPLES_DIR = DATA_DIR / "samples"
OPEN_METEO_RAW_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)


## Implementation
The event schema is intentionally flat so Spark can parse it with an explicit `StructType` in notebook `06`. The notebook always materializes local JSONL evidence. Kafka publishing is a guarded external integration step.

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
import json
import pandas as pd
import requests

OPEN_METEO_BASE_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
EVENTS_PATH = OPEN_METEO_RAW_DIR / "open_meteo_air_quality_events.jsonl"
MANIFEST_PATH = OPEN_METEO_RAW_DIR / "open_meteo_ingestion_manifest.json"
SAMPLE_EVENTS_PATH = SAMPLES_DIR / "open_meteo_phase5_events_sample.jsonl"
MOCK_BROKER_PATH = OPEN_METEO_RAW_DIR / "mock_kafka_air_quality_live.jsonl"
RUN_OPEN_METEO_API_FETCH = os.getenv("RUN_OPEN_METEO_API_FETCH", "true").lower() == "true"
ALLOW_CONTROLLED_OPEN_METEO_FALLBACK = os.getenv("ALLOW_CONTROLLED_OPEN_METEO_FALLBACK", "true").lower() == "true"
RUN_OPEN_METEO_KAFKA_PRODUCER = os.getenv("RUN_OPEN_METEO_KAFKA_PRODUCER", "false").lower() == "true"
BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "<kafka-host>:9092")
TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "LIVE-bdeng_gXX_air_quality_live")
KAFKA_MODE = os.getenv("KAFKA_MODE", "auto").lower()
ALLOW_KAFKA_MOCK_FALLBACK = os.getenv("ALLOW_KAFKA_MOCK_FALLBACK", "true").lower() == "true"
KAFKA_CONSUMER_TIMEOUT_MS = int(os.getenv("KAFKA_CONSUMER_TIMEOUT_MS", "10000"))
KAFKA_CONSUMER_MAX_MESSAGES = int(os.getenv("KAFKA_CONSUMER_MAX_MESSAGES", "8"))
OPEN_METEO_REQUEST_TIMEOUT_SECONDS = int(os.getenv("OPEN_METEO_REQUEST_TIMEOUT_SECONDS", "20"))
OPEN_METEO_MAX_HOURS_TO_SEND = int(os.getenv("OPEN_METEO_MAX_HOURS_TO_SEND", "1"))
assert KAFKA_MODE in {"auto", "kafka", "mock"}
assert OPEN_METEO_MAX_HOURS_TO_SEND >= 1

city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
required_city_columns = {"city_id", "city_name", "country_code", "latitude", "longitude"}
assert required_city_columns.issubset(city_reference_df.columns)
assert city_reference_df["city_id"].is_unique

def build_open_meteo_params(latitude: float, longitude: float) -> dict:
    return {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "pm2_5,pm10,nitrogen_dioxide",
        "timezone": "UTC",
        "forecast_days": 1,
    }

def controlled_open_meteo_payload(city_index: int) -> dict:
    times = [f"2026-01-15T{hour:02d}:00" for hour in range(24)]
    return {
        "hourly": {
            "time": times,
            "pm2_5": [float(6 + city_index + hour % 5) for hour in range(24)],
            "pm10": [float(12 + city_index + hour % 7) for hour in range(24)],
            "nitrogen_dioxide": [float(18 + city_index + hour % 9) for hour in range(24)],
        },
        "controlled_fallback": True,
    }

def load_open_meteo_payload(city_row: pd.Series, city_index: int) -> tuple[dict, str, str | None]:
    raw_path = OPEN_METEO_RAW_DIR / f"{city_row['city_id']}.json"
    if RUN_OPEN_METEO_API_FETCH:
        try:
            response = requests.get(
                OPEN_METEO_BASE_URL,
                params=build_open_meteo_params(city_row["latitude"], city_row["longitude"]),
                timeout=OPEN_METEO_REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            payload = response.json()
            raw_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
            return payload, "fetched_api", None
        except requests.RequestException as exc:
            fetch_error = str(exc)
            if raw_path.exists():
                return json.loads(raw_path.read_text(encoding="utf-8")), "loaded_local_bronze_after_api_error", fetch_error
            if ALLOW_CONTROLLED_OPEN_METEO_FALLBACK:
                payload = controlled_open_meteo_payload(city_index)
                raw_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
                return payload, "controlled_offline_fallback", fetch_error
            raise
    if raw_path.exists():
        return json.loads(raw_path.read_text(encoding="utf-8")), "loaded_local_bronze", None
    if ALLOW_CONTROLLED_OPEN_METEO_FALLBACK:
        payload = controlled_open_meteo_payload(city_index)
        raw_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        return payload, "controlled_offline_fallback", "API fetch disabled and no local Bronze JSON existed"
    raise FileNotFoundError(f"Missing local Bronze JSON: {raw_path}. Enable RUN_OPEN_METEO_API_FETCH.")

def build_air_quality_event(city_id: str, timestamp_utc: str, hourly_row: dict, ingestion_time_utc: str, data_status: str) -> dict:
    event_key = f"{city_id}|{timestamp_utc}|open_meteo|1.0"
    return {
        "event_id": sha256(event_key.encode("utf-8")).hexdigest(),
        "schema_version": "1.0",
        "source": "open_meteo",
        "city_id": city_id,
        "event_time_utc": timestamp_utc,
        "ingestion_time_utc": ingestion_time_utc,
        "data_status": data_status,
        "pm2_5": hourly_row.get("pm2_5"),
        "pm10": hourly_row.get("pm10"),
        "no2": hourly_row.get("nitrogen_dioxide"),
    }

def payload_to_events(city_id: str, payload: dict, ingestion_time_utc: str, data_status: str) -> list[dict]:
    hourly = payload.get("hourly", {})
    required_fields = ["time", "pm2_5", "pm10", "nitrogen_dioxide"]
    missing = [field for field in required_fields if field not in hourly]
    if missing:
        raise ValueError(f"Open-Meteo payload for {city_id} is missing hourly fields: {missing}")
    lengths = {field: len(hourly[field]) for field in required_fields}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"Open-Meteo hourly arrays for {city_id} have inconsistent lengths: {lengths}")
    complete_indexes = [index for index in range(lengths["time"]) if all(hourly[field][index] is not None for field in required_fields if field != "time")]
    if not complete_indexes:
        raise ValueError(f"Open-Meteo payload for {city_id} has no complete pollutant hour")
    selected_indexes = complete_indexes[-OPEN_METEO_MAX_HOURS_TO_SEND:]
    events = []
    for index in selected_indexes:
        timestamp = hourly["time"][index]
        hourly_row = {field: hourly[field][index] for field in required_fields if field != "time"}
        events.append(build_air_quality_event(city_id, f"{timestamp}:00Z", hourly_row, ingestion_time_utc, data_status))
    return events

ingestion_time_utc = datetime.now(timezone.utc).isoformat()
api_results = []
events = []
for city_index, (_, city_row) in enumerate(city_reference_df.iterrows()):
    payload, load_status, fetch_error = load_open_meteo_payload(city_row, city_index)
    data_status = "controlled_offline_fallback" if payload.get("controlled_fallback") else load_status
    city_events = payload_to_events(city_row["city_id"], payload, ingestion_time_utc, data_status)
    events.extend(city_events)
    raw_path = OPEN_METEO_RAW_DIR / f"{city_row['city_id']}.json"
    hourly = payload.get("hourly", {})
    missing_value_count = sum(value is None for field in ["pm2_5", "pm10", "nitrogen_dioxide"] for value in hourly.get(field, []))
    api_results.append({"city_id": city_row["city_id"], "city_name": city_row["city_name"], "status": data_status, "load_status": load_status, "http_status_code": 200 if load_status == "fetched_api" else None, "file_path": str(raw_path), "retrieved_at_utc": ingestion_time_utc, "event_count": len(city_events), "missing_value_count": missing_value_count, "fetch_error": fetch_error})

EVENTS_PATH.write_text("\n".join(json.dumps(event) for event in events) + "\n", encoding="utf-8")
events_df = pd.DataFrame(events)
api_results_df = pd.DataFrame(api_results)
MANIFEST_PATH.write_text(api_results_df.to_json(orient="records", indent=2), encoding="utf-8")
SAMPLE_EVENTS_PATH.write_text("\n".join(json.dumps(event) for event in events[:min(8, len(events))]) + "\n", encoding="utf-8")
api_results_df

## Validation / Quality Checks
Validate API coverage, stable event keys, timestamp shape, allowed sources, non-negative pollutant values where present, local JSONL round-trip, Kafka producer delivery, and bounded consumer evidence. Offline runs use the explicit mock broker.

In [ ]:
required_event_columns = {
    "event_id", "schema_version", "source", "city_id", "event_time_utc",
    "ingestion_time_utc", "data_status", "pm2_5", "pm10", "no2",
}
assert required_event_columns.issubset(events_df.columns)
assert set(events_df["city_id"]) == set(city_reference_df["city_id"])
assert events_df["event_id"].is_unique
assert (events_df["schema_version"] == "1.0").all()
assert (events_df["source"] == "open_meteo").all()
assert set(events_df["data_status"]).issubset({"fetched_api", "loaded_local_bronze", "loaded_local_bronze_after_api_error", "controlled_offline_fallback"})
assert events_df["event_time_utc"].str.endswith("Z").all()
assert events_df[["pm2_5", "pm10", "no2"]].apply(lambda values: values.dropna().ge(0).all()).all()

roundtrip_events = [json.loads(line) for line in EVENTS_PATH.read_text(encoding="utf-8").splitlines()]
assert len(roundtrip_events) == len(events)

from uuid import uuid4

def publish_and_consume_mock(events: list[dict]) -> tuple[dict, list[dict]]:
    MOCK_BROKER_PATH.write_text("\n".join(json.dumps(event) for event in events) + "\n", encoding="utf-8")
    consumed = [json.loads(line) for line in MOCK_BROKER_PATH.read_text(encoding="utf-8").splitlines()[:KAFKA_CONSUMER_MAX_MESSAGES]]
    return {"mode": "mock", "topic": TOPIC, "sent": len(events), "delivery_errors": 0, "mock_path": str(MOCK_BROKER_PATH)}, consumed

def publish_and_consume_kafka(events: list[dict]) -> tuple[dict, list[dict]]:
    if TOPIC in {"air_quality_live", "bdeng_gXX_air_quality_live", "LIVE-bdeng_gXX_air_quality_live"} or "<" in BOOTSTRAP_SERVERS:
        raise ValueError("Configure a reachable Kafka broker and a real group-specific topic before publishing.")
    from kafka import KafkaConsumer, KafkaProducer
    consumer = KafkaConsumer(TOPIC, bootstrap_servers=BOOTSTRAP_SERVERS, group_id=f"phase5-smoke-{uuid4().hex}", auto_offset_reset="latest", enable_auto_commit=False, consumer_timeout_ms=KAFKA_CONSUMER_TIMEOUT_MS, value_deserializer=lambda value: json.loads(value.decode("utf-8")))
    for _ in range(5):
        consumer.poll(timeout_ms=1000)
        if consumer.assignment():
            break
    if not consumer.assignment():
        consumer.close()
        raise RuntimeError("Kafka consumer could not obtain a topic partition assignment before publishing.")
    consumer.seek_to_end(*consumer.assignment())
    producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS, key_serializer=lambda value: value.encode("utf-8"), value_serializer=lambda value: json.dumps(value).encode("utf-8"))
    futures = [producer.send(TOPIC, key=event["city_id"], value=event) for event in events]
    for future in futures:
        future.get(timeout=20)
    producer.flush(timeout=20)
    producer.close()
    expected_ids = {event["event_id"] for event in events}
    consumed = []
    for message in consumer:
        if message.value.get("event_id") in expected_ids:
            consumed.append(message.value)
        if len(consumed) >= min(KAFKA_CONSUMER_MAX_MESSAGES, len(events)):
            break
    consumer.close()
    if not consumed:
        raise RuntimeError("Kafka consumer smoke test did not receive any event from this producer run.")
    return {"mode": "kafka", "topic": TOPIC, "sent": len(events), "delivery_errors": 0, "bootstrap_servers": BOOTSTRAP_SERVERS}, consumed

broker_result = None
consumed_events = []
fallback_reason = None
if RUN_OPEN_METEO_KAFKA_PRODUCER and KAFKA_MODE in {"auto", "kafka"}:
    try:
        broker_result, consumed_events = publish_and_consume_kafka(events)
    except Exception as exc:
        if KAFKA_MODE == "kafka" or not ALLOW_KAFKA_MOCK_FALLBACK:
            raise
        fallback_reason = str(exc)
        broker_result, consumed_events = publish_and_consume_mock(events)
else:
    broker_result, consumed_events = publish_and_consume_mock(events)

broker_result["consumed"] = len(consumed_events)
broker_result["fallback_reason"] = fallback_reason
assert broker_result["sent"] == len(events)
assert broker_result["delivery_errors"] == 0
assert len(consumed_events) >= 1
assert required_event_columns.issubset(consumed_events[0])
print({"events_path": str(EVENTS_PATH), "manifest_path": str(MANIFEST_PATH), "event_count": len(events), "broker": broker_result})
print({"consumer_sample": consumed_events[0]})
events_df.head()

## Results
The REST API path attempts all selected cities and creates local Bronze JSON plus a validated JSONL event batch. If network access fails, controlled fallback rows keep the mechanics reproducible and remain visibly labeled. With `RUN_OPEN_METEO_KAFKA_PRODUCER=true`, the same events are published to Kafka and verified by a bounded consumer smoke test. Local runs use the explicit mock broker.

## Limitations
Live API values are current context and must not be mixed with historical EEA conclusions without labeling. Controlled fallback rows and mock-broker delivery prove mechanics only and must not support analytical or FH-Kafka claims. Kafka delivery uses at-least-once semantics; downstream Spark processing must deduplicate by `event_id`.

## Next step
Run notebook `06` for Spark Structured Streaming from Kafka to Parquet.